In [5]:
import requests


response = requests.get("http://localhost:11434/api/tags")
print("Ollama is running!", response.json())


Ollama is running! {'models': [{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-06-18T10:27:15.42588938Z', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16', 'context_length': 2048, 'embedding_length': 768}, 'capabilities': ['embedding']}, {'name': 'qwen3:4b', 'model': 'qwen3:4b', 'modified_at': '2026-06-15T15:08:02.587308377Z', 'size': 2497293931, 'digest': '359d7dd4bcdab3d86b87d73ac27966f4dbb9f5efdfcc75d34a8764a09474fae7', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen3', 'families': ['qwen3'], 'parameter_size': '4.0B', 'quantization_level': 'Q4_K_M', 'context_length': 262144, 'embedding_length': 2560}, 'capabilities': ['completion', 'tools', 'thinking']}]}


In [6]:
from langchain_ollama import OllamaLLM, OllamaEmbeddings

llm = OllamaLLM(base_url="http://localhost:11434", model="llama3.2")
embedding_model = OllamaEmbeddings(base_url="http://localhost:11434", model="nomic-embed-text")

print("LLM and embeddings ready!")

LLM and embeddings ready!


In [7]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="qwen3:4b", base_url="http://localhost:11434")

response = llm.invoke("tell me a joke")
print(response)

Here's a quick, clean joke for you 😄:

**Why don't scientists trust atoms?**  
*Because they make up everything!*  

Let me know if you want another! 😊


RAG APPLICATION USING AUTHENTIC DATA

Preparing document


In [9]:
from langchain_core.documents import Document
my_text="""

"All the information about rare plants" is the botanical equivalent of asking for "all the information about the internet." Humans do enjoy assigning impossible scopes to tasks and then handing them to someone else.

Still, here's a comprehensive overview.

What Are Rare Plants?

A rare plant is a species that exists in small populations, has a limited geographic range, or faces significant threats that could lead to extinction.

A plant may be considered rare if it:

Exists in only a few locations
Has very small populations
Occupies a highly specialized habitat
Is declining rapidly
Is threatened by human activities or climate change

Rare does not always mean endangered, but many rare plants are endangered.

Why Do Plants Become Rare?
1. Habitat Loss

The biggest threat worldwide.

Examples:

Deforestation
Urban development
Mining
Agriculture expansion
2. Climate Change

Changes in temperature and rainfall can destroy habitats.

3. Overcollection

Collectors sometimes remove rare plants from the wild.

Examples:

Orchids
Carnivorous plants
Succulents
4. Invasive Species

Non-native plants may outcompete native species.

5. Pollution

Industrial waste and pesticides can damage ecosystems.

Types of Rare Plants
Endemic Plants

Found only in one specific location.

Example:

Rafflesia arnoldii
Relict Plants

Ancient species that survived from earlier geological periods.

Example:

Wollemi Pine
Carnivorous Plants

Obtain nutrients from insects.

Examples:

Venus Flytrap
Cobra Lily
Famous Rare Plants
1. Rafflesia arnoldii

Location: Indonesia and Malaysia

Features:

Largest single flower in the world
Can exceed 1 meter in diameter
Smells like rotting meat
Pollinated by flies
2. Wollemi Pine

Location: Australia

Features:

Thought extinct until 1994
Sometimes called a "living fossil"
Fewer than 100 mature trees remain in the wild
3. Ghost Orchid

Features:

Appears to float in air
Extremely difficult to cultivate
Pollinated by specialized moths
4. Middlemist's Red

Features:

One of the rarest flowering plants
Only a few known cultivated specimens exist"""
docs = [Document(page_content=my_text, metadata={"source": "Rare Plants", "Features": "Habitat, Threats, Conservation"})]
print(f"Loaded {len(docs)} document")



Loaded 1 document


In [10]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=80)
chunks = splitter.split_documents(docs)
print(f" Split into {len(chunks)} chunks ")
chunks

 Split into 4 chunks 


[Document(metadata={'source': 'Rare Plants', 'Features': 'Habitat, Threats, Conservation'}, page_content='"All the information about rare plants" is the botanical equivalent of asking for "all the information about the internet." Humans do enjoy assigning impossible scopes to tasks and then handing them to someone else.\n\nStill, here\'s a comprehensive overview.\n\nWhat Are Rare Plants?\n\nA rare plant is a species that exists in small populations, has a limited geographic range, or faces significant threats that could lead to extinction.\n\nA plant may be considered rare if it:\n\nExists in only a few locations\nHas very small populations\nOccupies a highly specialized habitat\nIs declining rapidly\nIs threatened by human activities or climate change'),
 Document(metadata={'source': 'Rare Plants', 'Features': 'Habitat, Threats, Conservation'}, page_content='Rare does not always mean endangered, but many rare plants are endangered.\n\nWhy Do Plants Become Rare?\n1. Habitat Loss\n\nThe

In [11]:

test_embedding = embedding_model.embed_query("Hello, this is a test.")
print(f"First 5 values: {test_embedding[:5]}")


First 5 values: [0.051687695, 0.026657073, -0.17383233, -0.032872576, 0.080336116]


In [13]:

from langchain_community.vectorstores import Chroma
vectorstore=Chroma.from_documents(documents=chunks,embedding=embedding_model)
print("Embeddings created and stored!")

Embeddings created and stored!


SEMANTIC Search

In [16]:
vectorstore.similarity_search("What are rare plants?", k=4)

[Document(metadata={'Features': 'Habitat, Threats, Conservation', 'source': 'Rare Plants'}, page_content='"All the information about rare plants" is the botanical equivalent of asking for "all the information about the internet." Humans do enjoy assigning impossible scopes to tasks and then handing them to someone else.\n\nStill, here\'s a comprehensive overview.\n\nWhat Are Rare Plants?\n\nA rare plant is a species that exists in small populations, has a limited geographic range, or faces significant threats that could lead to extinction.\n\nA plant may be considered rare if it:\n\nExists in only a few locations\nHas very small populations\nOccupies a highly specialized habitat\nIs declining rapidly\nIs threatened by human activities or climate change'),
 Document(metadata={'Features': 'Habitat, Threats, Conservation', 'source': 'Rare Plants'}, page_content='Rare does not always mean endangered, but many rare plants are endangered.\n\nWhy Do Plants Become Rare?\n1. Habitat Loss\n\nThe

Talk to LLM

In [18]:
context =vectorstore.similarity_search("What are rare plants?", k=4)
response=llm.invoke("tell me about rare plants based on this {context} ")
print(response)


I notice that **your query says "based on this {context}" but there is no actual context provided** in your message. The `{context}` placeholder is empty — it’s likely you forgot to include the specific context you’d like me to analyze (e.g., a text excerpt, region, scientific criteria, or research focus).

**To give you accurate, helpful information about rare plants, I need your context.** Here’s what I’d need to do it right:

### 🔍 What I need from you:
1. **Specify the context** (e.g.,):
   - A region (e.g., "Amazon rainforest," "California," "Japan")
   - A type of rarity (e.g., "endemic species," "critically endangered," "rare in cultivation")
   - A scientific classification (e.g., "plants with <100 known specimens," "species in the Asteraceae family")
   - A text excerpt or research focus (e.g., "plants from the 2023 IUCN Red List update")
   - Or even a simple phrase like *"rare plants in Australia"*

2. **Example of how to provide context** (to avoid confusion):  
   > *"Tell